https://knaaptime.com/urban_analysis/02_interpolation/dasymetric.html

#### interpolation of resident numbers from 2010 to 2021

Using Target Density Weighting (TDW) method found in the paper `Data-enriched Interpolation for Temporally Consistent Population Compositions`

1. Calculate subzone densities:
$Density = \frac{residents\ in\ subzone}{land area}$

2. Establish weigth ratio:

Assume the subzone population is proportional to the density of the year's total population. Calculate a `density ratio` for each subzone

$Ratio = \frac{Subzone\ Density}{National\ Average\ Density}$

3. Temporal Interpolation of Weights:

As we only have population data for 2010, 2015 and 2020, need to estimate the density eights for the missing years.

Assume that the subzone's density ratio moves linearly

4. Interpolate:

Multiply the annual national counts for each age group by the calculated weights. 

#### Interpolate resident numbers by age group
Census data for 2020 follows 2019 masterplan for geospatial data, 2015 follows 2014 masterplan, 2010 follows 2008 masterplan

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
from shapely.wkt import loads

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [3]:
# get subzone data with their corresponding areas in km^2
def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_boundary_{year}.gpkg")
    subzones_gpd = gpd.read_file(subzones_filepath)
    subzones_gpd = subzones_gpd.map(lambda s: s.lower() if isinstance(s, str) else s)
    # lower() the column names of subzones_gpd
    subzones_gpd.columns = subzones_gpd.columns.str.lower()    

    # there is no landuse data found for 2008 masterplan, 
    # so not able to merge with the subzone_classification file obtained from extracting_landuse_type.ipynb
    if year == "2008":
        return subzones_gpd

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones_csv = pd.read_csv(subzones_filepath)
    subzones_csv = subzones_csv.map(lambda s: s.lower() if isinstance(s, str) else s)

    subzones_with_geom = subzones_csv.merge(
        subzones_gpd[["subzone_n", "geometry"]],
        on = "subzone_n",
        how = "left"
    )

    missing_count = subzones_with_geom[subzones_with_geom.columns[-1]].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} subzones did not find a match in the CSV.")

    return subzones_with_geom

In [4]:
def get_demographic_data(year: int):

    year = str(year)

    demographic_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/{year}_age_group_planning_area_subzone.xlsx")
    demographic_df = pd.read_excel(demographic_filepath, sheet_name = "subzone")

    demographic_df = demographic_df.rename(columns = {
        "subzone": "subzone_n",
        "planning_area": "pln_area_n"
    })

    # 2015 census data has different column names, standardise it
    if year == "2015":
        old_cols = [
            'total_0_4', 'total_5_9', 'total_10_14', 'total_15_19', 'total_20_24', 
            'total_25_29', 'total_30_34', 'total_35_39', 'total_40_44', 'total_45_49', 
            'total_50_54', 'total_55_59', 'total_60_64', 'total_65_69', 'total_70_74', 
            'total_75_79', 'total_80_84', 'total_85andover'
        ]

        mapping = {}
        for col in old_cols:
            # Remove 'total_' prefix and replace underscores with ' - '
            new_name = col.replace('total_', '').replace('_', ' - ')
            
            # Handle the special '85andover' case you mentioned
            if '85andover' in new_name:
                mapping[col] = '85 - 89' # Based on your specific request
            else:
                mapping[col] = new_name

        mapping["total_85andover"] = "85 & Over"

        demographic_df = demographic_df.rename(columns = mapping)

    demographic_df["pln_area_n"] = demographic_df["pln_area_n"].ffill()

    ## the subzones for punggol is named differently in the masterplan for 2010 and 2015, 
    # instead of eg: subzone 2, it is sz2, so renaming it
    if year == "2015" or year == "2010":
        # mask_pa = demographic_df["pln_area_n"].str.lower() == "punggol"
        
        # # Mapping dictionary to make this cleaner
        # sz_map = {
        #     "subzone 2": "sz2",
        #     "subzone 3": "sz3",
        #     "subzone 4": "sz4",
        #     "subzone 5": "sz5"
        # }
        
        # for old_sz, new_sz in sz_map.items():
        #     demographic_df.loc[
        #         (demographic_df["subzone_n"].str.lower() == old_sz) & mask_pa, 
        #         "subzone_n"
        #     ] = new_sz

        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 2") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz2"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 3") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz3"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 4") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz4"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 5") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz5"

    return demographic_df

# Step 1 : Calculate population density per subzone

In [5]:
subzone_df = get_subzone_data(2008)
subzone_df.head(3)

,objectid,subzone_no,subzone_n,subzone_c,ca_ind,pln_area_n,pln_area_c,region_n,region_c,inc_crc,fmel_upd_d,x_addr,y_addr,shape_leng,shape_area,geometry
0,1,1,woodlands regional centre,wdsz01,n,woodlands,wd,north region,nr,8877712517deec59,2016-05-11,22571.2094,46698.8735,3254.343275,5.956521e+05,"MULTIPOLYGON (((22629.486 46980.246, 22683.012..."
1,2,1,kranji,sksz01,n,sungei kadut,sk,north region,nr,de75f42ec0f86b2b,2016-05-11,19767.5129,46226.3625,7906.523525,3.654121e+06,"MULTIPOLYGON (((20329.947 47234.988, 20333.366..."
2,3,2,turf club,sksz02,n,sungei kadut,sk,north region,nr,5d555db858522781,2016-05-11,20234.1723,44507.1299,7677.244752,3.290816e+06,"MULTIPOLYGON (((21094.367 44815.121, 21099.854..."


In [6]:
# convert the Pandas DataFrame back to a GeoDataFrame
# We explicitly pass the geometry column
if not isinstance(subzone_df, gpd.GeoDataFrame):
    subzone_df = gpd.GeoDataFrame(subzone_df, geometry='geometry')

# ensure the original data has the correct starting CRS (WGS84)
# (Only needed if it isn't already set; usually .gpkg files have this metadata)
if subzone_df.crs is None:
    subzone_df.set_crs(epsg=4326, inplace=True)

# transform the entire GeoDataFrame to SVY21 (Meters)
subzone_df_meters = subzone_df.to_crs(epsg=3414)

# calculate area
subzone_df['area_sqm'] = subzone_df_meters.geometry.area
subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000

subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

# display(subzone_df[['subzone_n', 'area_sqkm']].head())
display(subzone_areas)

,subzone_n,pln_area_n,area_sqm,area_sqkm,geometry
0,woodlands regional centre,woodlands,5.956521e+05,0.595652,"MULTIPOLYGON (((22629.486 46980.246, 22683.012..."
1,kranji,sungei kadut,3.654121e+06,3.654121,"MULTIPOLYGON (((20329.947 47234.988, 20333.366..."
2,turf club,sungei kadut,3.290816e+06,3.290816,"MULTIPOLYGON (((21094.367 44815.121, 21099.854..."
3,nee soon,yishun,2.209311e+06,2.209311,"MULTIPOLYGON (((26860.555 43913.328, 26839.719..."
4,yishun west,yishun,1.517705e+06,1.517705,"MULTIPOLYGON (((28228.609 45792.488, 28223.039..."
...,...,...,...,...,...
306,sungei serangoon west,sengkang,1.572812e+06,1.572812,"MULTIPOLYGON (((37009.969 40867.361, 37011.087..."
307,sungei serangoon,hougang,1.333531e+06,1.333531,"MULTIPOLYGON (((36538.391 39761.352, 36517.439..."
308,hougang central,hougang,2.291220e+06,2.291220,"MULTIPOLYGON (((35948.803 40264.817, 35951.927..."
309,changi west,changi,4.849019e+06,4.849019,"MULTIPOLYGON (((45347.86 41116.219, 45350.666 ..."


In [7]:
demographic_df = get_demographic_data(2010)
demographic_df

,pln_area_n,subzone_n,total,0 - 4,5 - 9,10 - 14,15 - 19,20 - 24,25 - 29,30 - 34,35 - 39,40 - 44,45 - 49,50 - 54,55 - 59,60 - 64,65 & Over,total_above_60
0,total,total,3771721,194432,215675,244302,263750,247190,272639,298687,320024,309441,323459,303044,248696,191995,338387,530382
1,ang mo kio,total,179297,7967,8424,9335,10457,10656,13400,14502,14510,13525,14862,14605,13785,11868,21401,33269
2,ang mo kio,cheng san,30503,1334,1387,1428,1587,1720,2471,2691,2569,2365,2471,2461,2487,2099,3433,5532
3,ang mo kio,chong boon,29903,1254,1304,1429,1563,1713,2348,2469,2326,2241,2431,2388,2380,2137,3920,6057
4,ang mo kio,kebun bahru,25854,1089,1189,1327,1418,1460,1901,2076,2123,1968,2122,2078,1992,1752,3359,5111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,yishun,yishun central,1608,67,45,102,161,144,147,116,79,109,161,152,124,76,125,201
224,yishun,yishun east,32933,1729,2029,2385,2975,2579,2353,2476,2642,2755,3244,2725,2018,1302,1721,3023
225,yishun,yishun south,40551,1842,2133,2488,3252,3406,3228,3008,3150,3033,3830,3491,2849,2009,2832,4841
226,yishun,yishun west,60219,2920,3222,3644,3916,4614,4987,4685,4816,4993,5408,5157,4189,2904,4764,7668


#### calculate the density of each subzone
Density (resident per $km^2$) = $\frac{residents\ in\ subzone}{land\ area\ km^2}$

####  each set of census data has different recordings of age group numbers. I have chosen to preserve them instead of combining ages 65 and over. This is because granularity will help with the accuracy of the Target Density Weighting interpolation

In [8]:
def calculate_density(demographic_df, subzone_areas, year:int):
    
    year = str(year)


    if year == "2010":
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 & Over', 'total_above_60']
    
    if year == "2015":    
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 & Over', 'total_above_60']
        
    if year == "2020":
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 - 89', '90 & over', 'total_above_60']
        
    demographic_df = demographic_df[age_grp_columns].copy()

    # standardize casing and remove hidden whitespace
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()

    density_df = subzone_areas.merge(
        demographic_df,
        on = "subzone_n",
        how = "left"
    )
    
    for age_group in age_grp_columns[2 : -1]:
        new_column_name = "density_" + age_group

        density_df[f"density_{age_group}"] =  density_df[age_group] / density_df["area_sqkm"]

    return density_df
    

In [9]:
density_df = calculate_density(demographic_df, subzone_areas, 2010)
# fill NaN results with 0 (0 population data for that subzone)
# density_df = density_df.fillna(0)
save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2008/pop_density_2010.csv")
density_df.to_csv(save_to_file)

In [10]:
subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
# subzone_areas.to_csv("subzone_areas.csv")
demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()
# demographic_df.to_csv("demographic_df.csv")

check_df = subzone_areas[['subzone_n', 'pln_area_n']].merge(
    demographic_df[['subzone_n', 'pln_area_n']], 
    on=['subzone_n', 'pln_area_n'],
    how='outer', 
    indicator=True
)

failed_df = check_df[check_df['_merge'] == 'right_only'][['subzone_n', 'pln_area_n']]
print(failed_df.to_string(index=False))

subzone_n      pln_area_n
    total      ang mo kio
    total           bedok
    total          bishan
    total     bukit batok
    total     bukit merah
    total   bukit panjang
    total     bukit timah
    total          changi
    total   choa chu kang
    total        clementi
    total   downtown core
    total         geylang
    total         hougang
    total     jurong east
    total     jurong west
    total         kallang
    total          mandai
    total   marine parade
    total          newton
    total          novena
    total          others
    total          outram
    total       pasir ris
    total         punggol
    total      queenstown
    total    river valley
    total          rochor
    total       sembawang
    total        sengkang
    total       serangoon
    total singapore river
    total        tampines
    total         tanglin
    total       toa payoh
    total           total
    total       woodlands
    total          yishun


In [ ]:
search_list = ['jalan kayu east', 'jurong lake', 'jalan kayu west']

# Returns a boolean Series (True/False for every row)
mask = subzone_df['subzone_n'].isin(search_list)

# Use the mask to see the actual rows
found_rows = subzone_df[mask]
found_rows

# Step 2: Calculate Density Ratio per subzone

#### First we need to obtain the national average density for each age group

In [ ]:
density_cols = [col for col in density_df.columns if "density" in col.lower()]
density_cols